In [3]:

"""
HFEv- (Hidden Field Equations with Vinegar and Minus) Dijital İmzalama Şeması
==============================================================================

Bu modül, çok değişkenli polinom tabanlı (Multivariate Quadratic - MQ)
kriptografinin BigField ailesine ait HFEv- imza şemasının uçtan uca bir
referans implementasyonunu içermektedir. Modül; anahtar üretimi, imzalama
ve doğrulama olmak üzere üç temel kriptografik süreci ayrıntılı biçimde
yürütür ve her adımda ara sonuçları ekrana yazdırarak algoritmanın iç
işleyişinin izlenebilir olmasını sağlar.

Teorik Arka Plan
-----------------
HFE (Hidden Field Equations) ailesi, Patarin tarafından Matsumoto-Imai (MI)
şemasına yöneltilen doğrusallaştırma (linearization) saldırısına karşı bir
yanıt olarak önerilmiştir. Temel fikir, F_q asal cismi üzerinde n boyutlu bir
uzayda çalışmak yerine, F_{q^m} genişleme cismi üzerinde tek değişkenli bir
"merkez polinom" F(X) tanımlamak ve bu polinomun köklerinin verimli biçimde
bulunabilmesini sağlayacak şekilde derecesini D parametresi ile sınırlamaktır.
X^{q^i + q^j} biçimindeki terimler, F_{q^m} üzerinde bakıldığında yüksek
dereceli görünseler dahi, F_q-doğrusal Frobenius otomorfizmi (x -> x^q)
sayesinde F_q üzerine indirgendiklerinde yalnızca KUADRATİK (ikinci dereceden)
terimler üretirler. Bu, HFE ailesinin "gizli kuadratiklik" özelliğinin temelini
oluşturur ve açık anahtarın neden ikinci dereceden çok değişkenli bir polinom
sistemi olarak ortaya çıktığını açıklar.

HFEv- şeması, temel HFE yapısına iki bağımsız modifikasyon ekleyerek
güvenliğini artırır:

    1. Vinegar (v) Modifikasyonu:
       Merkez polinoma, Oil değişkenlerine (X'in temsil ettiği genişleme
       cismi elemanına) ek olarak, F_q üzerinde tanımlı v adet ekstra
       "vinegar" değişkeni eklenir. Bu değişkenler merkez dönüşümün
       cebirsel yapısını bozarak MinRank tabanlı saldırıları zorlaştırır.
       Bu şema Oil-Vinegar felsefesini BigField ailesine taşır.

    2. Minus (-) Modifikasyonu:
       Açık anahtardan rastgele seçilen 'a' adet denklem silinir (S dönüşümü
       ile boyut düşürülerek gerçekleştirilir). Bu, saldırganın gördüğü
       denklem sayısını azaltarak sistemi "eksik belirlenmiş" (underdetermined)
       hale getirir ve yapısal saldırıları zorlaştırır.

Bu iki modifikasyonun birlikte uygulanması "HFEv-" adlandırmasını doğurur ve
NIST Post-Kuantum Kriptografi standardizasyon sürecine aday gösterilen
QUARTZ, Gui ve GeMSS gibi imza şemalarının temelini oluşturur.

Algoritmanın Genel Akışı
-------------------------
1. Anahtar Üretimi:
   - Gizli afin dönüşümler T (n x n, tersinir) ve S ((m-a) x m, tam ranklı)
     rastgele seçilir.
   - F_{q^m} genişleme cismi üzerinde gizli bir HFEv merkez polinomu
     P(X, v_1, ..., v_v) rastgele katsayılarla inşa edilir.
   - Açık anahtar, P = S o Fbar o T bileşke dönüşümünün SEMBOLİK olarak
     hesaplanmasıyla elde edilir; burada Fbar, genişleme cismi elemanını
     F_q üzerinde n değişkenli koordinat polinomlarına geri döndüren
     izomorfizmadır.

2. İmzalama:
   - Mesaj özeti h, S dönüşümünün tersi alınarak (özel çözüm + çekirdek
     uzayından rastgele bir eleman) genişletilir ve genişleme cismindeki
     hedef değer W elde edilir.
   - Rastgele vinegar değerleri seçilerek merkez polinom tek değişkenli
     bir polinoma indirgenir ve bu polinomun kökleri (Sage'in yerleşik
     .roots() metodu; teorik olarak Berlekamp algoritması gibi sonlu
     cisimler üzerinde çarpanlara ayırma teknikleri) hesaplanır.
   - Kök bulunamazsa, farklı vinegar/minus rastgeleliği ile deneme
     tekrarlanır (retry mekanizması).
   - Bulunan kök, T dönüşümünün tersi ile orijinal imza uzayına geri
     taşınır.

3. Doğrulama:
   - Đmza, açık anahtar polinomlarında doğrudan yerine konularak P(z)
     hesaplanır ve mesaj özeti h ile karşılaştırılır. Açık anahtarla
     çalışıldığı için doğrulama, gizli anahtarın aksine çok hızlıdır.

Bu implementasyon; yüksek lisans tezi kapsamında HFEv- şemasının kavramsal
ve matematiksel yapısını somutlaştırmak amacıyla eğitim/araştırma amaçlı
geliştirilmiştir. Prodüksiyon ortamlarında kullanılmak üzere tasarlanmamış
olup sabit zamanlı (constant-time) işlemler veya yan kanal saldırılarına
karşı önlemler içermemektedir.

Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
"""

from sage.all import *


class HFEvminus:
    """
    HFEv- (HFE - Vinegar - Minus) dijital imzalama şemasının uçtan uca
    referans implementasyonu.

    Bu sınıf; gizli/açık anahtar üretimi, imzalama ve doğrulama süreçlerini
    kapsayan tam bir HFEv- yaşam döngüsünü modelleyen durumsal (stateful)
    bir nesne sunar. Sınıfın her bir örneği; sabit (q, m, D, v, a)
    parametre setine bağlı olarak kendi cisim genişlemesini, polinom
    halkalarını ve (üretildiğinde) gizli/açık anahtar bileşenlerini üzerinde
    taşır.

    Matematiksel Kurulum
    ---------------------
    - F_q  : Temel sonlu cisim (karakteristik/asal güç q).
    - F_{q^m} (self.E) : F_q'nun m. dereceden cisim genişlemesi; HFEv
      merkez polinomunun tanımlı olduğu "büyük cisim" (BigField).
    - R = F_q[x_0, ..., x_{n-1}] : Açık anahtar polinomlarının yaşadığı
      n = m + v değişkenli çok değişkenli polinom halkası.
    - R_E = F_{q^m}[X] : Merkez HFEv polinomunun tanımlı olduğu tek
      değişkenli polinom halkası (genişleme cismi katsayılı).

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (F_q). Genellikle küçük bir
        asal sayı veya asal kuvveti olarak seçilir.
    m : int
        Cisim genişlemesinin derecesi; aynı zamanda "Oil" değişkenlerinin
        (genişleme cismi elemanının F_q-koordinat sayısı) miktarını belirler.
    D : int
        HFE merkez polinomunun izin verilen maksimum derece sınırı.
        Bu sınır, X^{q^i+q^j} biçimindeki terimlerin oluşturulmasında
        q^i + q^j <= D koşulu ile kontrol edilir; D büyüdükçe merkez
        polinomun cebirsel karmaşıklığı (ve dolayısıyla kök bulma maliyeti)
        artar.
    v : int
        Vinegar değişkenlerinin sayısı. Bu değişkenler F_q üzerinde
        tanımlıdır ve merkez polinomun katsayılarını (doğrusal ve
        kuadratik biçimde) etkiler.
    a : int
        Minus modifikasyonunda açık anahtardan çıkarılacak (silinecek)
        denklem sayısı. 0 <= a < m koşulunu sağlamalıdır.

    Öznitelikler (Attributes)
    --------------------------
    n : int
        Toplam değişken sayısı (n = m + v); açık anahtarın girdi uzayının
        boyutu ve gizli T dönüşümünün tanım kümesinin boyutu.
    F : FiniteField
        Temel sonlu cisim F_q.
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    E : FiniteField
        F_q'nun m. dereceden genişlemesi (BigField), merkez polinomun
        katsayılarının ve köklerinin yaşadığı cisim.
    R_E : PolynomialRing
        E katsayılı tek değişkenli polinom halkası; merkez HFEv polinomu
        bu halkada tanımlanır.
    Secret_Map : dict
        Gizli merkez polinomun bileşenlerini (core, linear, gamma) saklayan
        sözlük. generate_keys() içinde doldurulur.
    T_mat, T_vec : Matrix, vector
        Gizli afin dönüşüm T(x) = T_mat * x + T_vec (n x n, tersinir).
    S_mat, S_vec : Matrix, vector
        Gizli afin dönüşüm S(z) = S_mat * z + S_vec ((m-a) x m, tam ranklı);
        Minus modifikasyonunu gerçekleştiren boyut düşürücü dönüşüm.
    Public_Key : vector
        Açık anahtarı oluşturan (m - a) adet çok değişkenli kuadratik
        polinomdan oluşan vektör.

    Notlar
    ------
    Kurucu metot (__init__), parametre geçerliliğini denetler ve gerekli
    tüm cebirsel yapıları (cisimler, polinom halkaları, cisim genişlemesi)
    anahtar üretiminden önce hazırlar. Böylece generate_keys(), sign() ve
    verify() metotları bu ortak altyapıyı tekrar tekrar kurmak
    zorunda kalmadan kullanabilir.
    """

    def __init__(self, q, m, D, v, a):
        """
        HFEv- şemasının parametrelerini doğrular ve temel cebirsel
        altyapıyı (sonlu cisim, polinom halkaları, cisim genişlemesi)
        kurar.

        Parametreler
        ------------
        q : int
            Temel sonlu cismin eleman sayısı.
        m : int
            Cisim genişleme derecesi (Oil/genişleme boyutu).
        D : int
            HFE merkez polinomunun maksimum derece sınırı (D >= 2).
        v : int
            Vinegar değişken sayısı (v >= 0).
        a : int
            Minus parametresi; silinecek denklem sayısı (0 <= a < m).

        İstisnalar (Raises)
        --------------------
        ValueError
            a, v veya D parametreleri geçerlilik koşullarını
            (0 <= a < m, v >= 0, D >= 2) sağlamadığında fırlatılır.

        Notlar
        ------
        Bu metot yalnızca YAPISAL kurulumu (cisimler, halkalar) gerçekleştirir;
        gizli/açık anahtarların üretimi generate_keys() metoduna bırakılmıştır.
        Bu ayrım, aynı parametre setiyle birden fazla anahtar üretimi
        denemesi yapılabilmesine imkân tanır.
        """
        self.q = q
        self.m = m # Oil değişkenleri / Genişleme derecesi
        self.D = D
        self.v = v # Vinegar değişkenleri
        self.a = a # Minus (çıkarılan denklem sayısı)
        self.n = m + v # Toplam değişken (Input boyutu)

        # --- Parametre Geçerlilik Denetimleri ---
        # Minus parametresi a, açık anahtardan en az bir denklemin
        # korunmasını (a < m) ve negatif olmayan bir silme miktarını
        # (a >= 0) garanti etmelidir; aksi halde S dönüşümü tanımsız kalır.
        if not (0 <= a < m):
            raise ValueError("Minus parametresi a, 0 <= a < m koşulunu sağlamalıdır.")

        # Vinegar değişken sayısı negatif olamaz; v = 0 seçilirse şema
        # klasik (Vinegar'sız) HFE-Minus şemasına indirgenir.
        if v < 0:
            raise ValueError("Vinegar değişken sayısı v negatif olamaz.")

        # D < 2 durumunda merkez polinom sabit veya doğrusal kalır ve
        # kriptografik olarak anlamlı bir kuadratik yapı elde edilemez.
        if D < 2:
            raise ValueError("HFE derece sınırı D en az 2 olmalıdır.")

        # 1. Temel Cisim ve Halka
        # F_q temel sonlu cismi; tüm açık anahtar aritmetiği bu cisim
        # üzerinde yürütülür.
        self.F = GF(q, 'a')
        # n = m + v değişkenli, F_q katsayılı çok değişkenli polinom halkası;
        # açık anahtarın ve T dönüşümünün tanımlandığı temel uzay.
        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = self.R.gens()

        # 2. Genişleme Cismi
        # Önce F_q üzerinde tek değişkenli bir yardımcı halka kurulur,
        # ardından bu halka üzerinde m. dereceden indirgenemez bir polinom
        # seçilerek F_{q^m} genişleme cismi (BigField) inşa edilir. Bu
        # genişleme, HFEv merkez polinomunun "gizli" tanım kümesidir ve
        # saldırgan tarafından bilinmemektedir (yalnızca anahtar sahibi bilir).
        self.R_uni = PolynomialRing(self.F, 'X')
        self.irr_poly = self.R_uni.irreducible_element(m)
        self.E = self.F.extension(self.irr_poly, 'w')
        # R_E: E katsayılı tek değişkenli polinom halkası; merkez HFEv
        # polinomu F(X, v_1, ..., v_v) bu halka üzerinde ifade edilir.
        self.R_E = PolynomialRing(self.E, 'X')

        print("\n" + "="*80)
        print(f"HFEv- (HFE - Vinegar - Minus) TAM DETAYLI ANALİZ")
        print("="*80)
        print(f"[KURULUM] Parametreler: q={q}, m={m}, v={v}, a={a}, D={D}")
        print(f"[KURULUM] İndirgenemez Polinom: {self.irr_poly}")
        print("-" * 80)

    def extension_to_vector(self, element):
        """
        Genişleme cismi F_{q^m} elemanını, F_q üzerindeki koordinat
        vektörüne (uzunluk m) dönüştürür.

        Teorik Gerekçe
        ---------------
        F_{q^m}, F_q üzerinde {1, w, w^2, ..., w^{m-1}} bazına sahip
        m-boyutlu bir vektör uzayı olarak görülebilir (burada w,
        indirgenemez polinomun bir kökü, yani genişleme cisminin
        üreteci). Bu metot, genişleme cismindeki bir elemanı bu baza
        göre katsayılarına ayırarak F_q^m vektör uzayına taşır; böylece
        Fbar: F_{q^m} -> F_q^m izomorfizmasının somut hesaplamasını
        gerçekleştirir.

        Parametreler
        ------------
        element : FiniteFieldElement
            E = F_{q^m} cisminde bir eleman (veya bu cisme gömülebilen
            bir polinom/eleman).

        Dönüş Değeri
        ------------
        vector
            F_q üzerinde uzunluğu m olan katsayı vektörü; element'in
            {1, w, ..., w^{m-1}} bazındaki temsili.
        """
        try: poly = element.lift()
        except:
            try: poly = element.polynomial()
            except: poly = element
        poly = self.R_uni(poly)
        coeffs = poly.list()
        # Genişleme derecesinden düşük dereceli katsayı listeleri, eksik
        # yüksek dereceli terimlerin katsayısı sıfır kabul edilerek m
        # uzunluğuna tamamlanır.
        while len(coeffs) < self.m: coeffs.append(self.F(0))
        return vector(self.F, coeffs)

    def vector_to_extension(self, vec):
        """
        F_q üzerindeki uzunluğu m olan bir koordinat vektörünü, genişleme
        cismi F_{q^m}'in karşılık gelen elemanına dönüştürür.

        Bu metot extension_to_vector() işleminin TERSİDİR: verilen katsayı
        vektörü {1, w, w^2, ..., w^{m-1}} bazına göre doğrusal
        kombinasyon olarak yeniden bir araya getirilir; yani
        Fbar^{-1}: F_q^m -> F_{q^m} izomorfizmasını hesaplar. İmzalama
        sürecinde, S dönüşümünün tersinden elde edilen F_q^m
        vektörünü genişleme cismindeki hedef değere (W) dönüştürmek
        için kullanılır.

        Parametreler
        ------------
        vec : vector
            F_q üzerinde uzunluğu m olan koordinat vektörü.

        Dönüş Değeri
        ------------
        FiniteFieldElement
            E = F_{q^m} cisminde, vec'in temsil ettiği eleman.
        """
        w_gen = self.E.gen()
        res = self.E(0)
        for i in range(self.m):
            res += vec[i] * (w_gen**i)
        return res

    def reduce_F_ideal(self, polys):
        """
        Verilen polinom vektörünü, "cisim denklemleri" (field equations)
        ideali x_i^q - x_i = 0 (i = 0, ..., n-1) altında indirger.

        Teorik Gerekçe
        ---------------
        F_q sonlu cisminin her elemanı x, x^q = x özdeşliğini sağlar
        (Fermat'nın küçük teoreminin genellemesi). Bu nedenle, F_q üzerinde
        tanımlı bir fonksiyon olarak x_i^q ile x_i birbirinin yerine
        geçebilir; bu ideal ile indirgeme yapılarak polinomların gereksiz
        yüksek dereceli (ancak F_q üzerinde x_i'ye eşdeğer) terimlerden
        arındırılması sağlanır. Bu adım, açık anahtar polinomlarının
        sahici (mümkün olan en düşük dereceden, burada kuadratik)
        temsiline ulaşmak için zorunludur.

        Parametreler
        ------------
        polys : iterable
            R halkasında tanımlı polinomlardan oluşan bir dizi/liste.

        Dönüş Değeri
        ------------
        vector
            Her bir girdi polinomunun cisim ideali FieldIdeal'e göre
            indirgenmiş hâlini içeren vektör.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)
        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    def generate_keys(self):
        """
        HFEv- şemasının gizli anahtarını (T, S dönüşümleri ve gizli merkez
        polinom) rastgele üretir ve bunlardan açık anahtarı (Public_Key)
        sembolik hesaplama yoluyla türetir.

        Algoritmanın Adımları
        -----------------------
        1. Afin Dönüşümler:
           - T: F_q^n -> F_q^n, tersinir bir doğrusal dönüşüm + rastgele
             öteleme vektörü (n x n tam ranklı matris).
           - S: F_q^m -> F_q^{m-a}, tam ranklı (m-a) x m matris + öteleme
             vektörü. Bu dönüşüm, boyut düşürerek Minus modifikasyonunu
             gerçekleştirir (m adet ham denklemden a tanesi "gizlenmiş"
             olur).

        2. Gizli HFEv Merkez Polinomu:
           Genişleme cismi E üzerinde, aşağıdaki üç bileşenin toplamından
           oluşan bir polinom F(X, v_1, ..., v_v) inşa edilir:

             a) Core (Çekirdek) Kısmı:
                Yalnızca X'e bağlı, klasik HFE terimleri X^{q^i+q^j}
                (q^i + q^j <= D koşuluyla sınırlı). Bu terimler, F_{q^m}
                üzerinde yüksek dereceli görünmelerine rağmen Frobenius
                otomorfizmasının F_q-doğrusallığı sayesinde F_q üzerine
                indirgendiklerinde KUADRATİK hale gelir.

             b) Linear (Doğrusal) X Kısmı:
                X^{q^i} teriminin katsayısı, vinegar değişkenlerine
                doğrusal bağlı bir ifadedir: (C + c_1*v_1 + ... + c_v*v_v).
                Bu terimler indirgendiğinde X (Oil) ile v (Vinegar)
                arasında KARIŞIK kuadratik terimler (Oil*Vinegar) üretir;
                bu, Vinegar modifikasyonunun cebirsel yapıyı bozan temel
                mekanizmasıdır.

             c) Gamma (Sabit Terim) Kısmı:
                Yalnızca vinegar değişkenlerine bağlı, sabit + doğrusal +
                kuadratik terimlerden oluşan bir ifade. İndirgendiğinde
                yalnızca Vinegar*Vinegar biçiminde kuadratik terimler
                üretir.

        3. Açık Anahtar İnşası (Sembolik):
           - x değişkenleri T dönüşümü ile y = T(x)'e taşınır.
           - y vektörünün ilk m bileşeni, genişleme cismi elemanı
             X_sembolik = sum(y_k * w^k) olarak birleştirilir (Oil kısmı);
             kalan v bileşen ise Vinegar değişkenleri V_sembolik olarak
             ayrıştırılır.
           - Gizli merkez polinom P, bu sembolik X ve V ifadeleriyle
             yeniden değerlendirilerek Z_sym = P(X_sembolik, V_sembolik)
             elde edilir (BÜYÜK CİSİM üzerinde sembolik bileşke).
           - Z_sym, cisim denklemleri (x^q - x) idealine göre indirgenerek
             en fazla kuadratik dereceye sahip Z_reduced elde edilir; bu,
             HFE ailesinin "gizli kuadratiklik" özelliğinin somut kanıtıdır.
           - Z_reduced, extension_to_vector() aracılığıyla F_q katsayılı
             m adet çok değişkenli polinoma (z_vec) geri dönüştürülür.
           - Son olarak S dönüşümü uygulanarak (Minus modifikasyonu ile
             boyut m'den m-a'ya düşürülerek) açık anahtar elde edilir:
             Public_Key = S(z_vec) = S_mat * z_vec + S_vec, ardından cisim
             idealine göre son bir indirgeme yapılır.

        Dönüş Değeri
        ------------
        vector
            (m - a) adet çok değişkenli kuadratik polinomdan oluşan açık
            anahtar (self.Public_Key olarak da saklanır).

        Yan Etkiler
        -----------
        self.T_mat, self.T_vec, self.S_mat, self.S_vec, self.Secret_Map
        ve self.Public_Key öznitelikleri bu metot tarafından doldurulur.
        Ayrıca süreç boyunca ayrıntılı ara çıktılar ekrana yazdırılır.
        """
        print("\n" + ">>> BÖLÜM 1: GİZLİ ANAHTAR ÜRETİMİ (KEY GENERATION) <<<")

        # --- 1. AFİN DÖNÜŞÜMLER ---
        # T dönüşümü: n x n boyutunda TERSİNİR rastgele bir matris.
        # Tersinirlik, T'nin bir bijeksiyon olmasını (dolayısıyla imzalama
        # sırasında T^-1'in var olmasını) garanti eder.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)

        # S dönüşümü: (m-a) x m boyutunda TAM RANKLI (rank = m-a) rastgele
        # bir matris. Bu, Minus modifikasyonunu gerçekleştiren boyut
        # düşürücü dönüşümdür; tam rank koşulu, S'nin örten olmasını ve
        # dolayısıyla her mesaj özeti için bir ön-görüntünün var
        # olabilmesini garanti eder.
        while True:
            self.S_mat = random_matrix(self.F, self.m - self.a, self.m)
            if self.S_mat.rank() == self.m - self.a: break
        self.S_vec = random_vector(self.F, self.m - self.a)

        print(f"\n[ADIM 1] Gizli Dönüşümler (S ve T)")
        print(f"   T Matrisi ({self.n}x{self.n}):\n{self.T_mat}")
        print(f"   T Vektörü (c_T): {self.T_vec}") # <--- EKLENDİ

        print(f"   S Matrisi ({self.m-self.a}x{self.m} - Minus!):\n{self.S_mat}")
        print(f"   S Vektörü (c_S): {self.S_vec}") # <--- EKLENDİ

        # --- 2. GİZLİ HFEv POLİNOMU ---
        # F(X, v) = Core(X) + Linear(v)*X^q^i + Quadratic(v)
        # Bu üç bileşen aşağıda ayrı ayrı inşa edilir; her biri farklı bir
        # cebirsel rolü (klasik HFE çekirdeği, Oil-Vinegar etkileşimi,
        # saf Vinegar terimleri) temsil eder.

        self.Secret_Map = { 'core': self.R_E(0), 'linear': [], 'gamma_quad': [] }
        X = self.R_E.gen()

        print(f"\n[ADIM 2] Gizli HFEv Polinomu F(X, V) Bileşenleri")

        # A) Core Kısım (Sadece X'li terimler)
        # Klasik HFE çekirdeği: X^{q^i + q^j} terimleri, q^i + q^j <= D
        # kısıtıyla sınırlandırılır. Bu sınırlama, merkez polinomun
        # F_{q^m} üzerindeki (görünürdeki) derecesini kontrol altında
        # tutarak kök bulma adımının hesaplama açısından uygulanabilir
        # kalmasını sağlar; D büyüdükçe güvenlik potansiyel olarak artar
        # ancak imzalama maliyeti de artar.
        print("   --- A) CORE KISMI (Sadece X'e bağlı terimler) ---")
        for i in range(self.m):
            for j in range(i + 1):
                if self.q**i + self.q**j <= self.D:
                    coeff = self.E.random_element()
                    if coeff != 0:
                        self.Secret_Map['core'] += coeff * (X**(self.q**i + self.q**j))
        print(f"   F_core(X) = {self.Secret_Map['core']}")

        # B) Linear X Kısım
        # X^(q^i) katsayısı: (C + c1*v1 + ... + cv*vv)
        # Bu katsayı, X'in (Oil) her kuvveti için VİNEGAR değişkenlerine
        # doğrusal bağlı bir ifadedir. İndirgeme sonrasında bu terimler
        # Oil*Vinegar biçiminde karışık kuadratik terimler üreterek
        # merkez dönüşümün cebirsel yapısını, saf HFE'ye kıyasla daha az
        # öngörülebilir hale getirir.
        print("\n   --- B) LINEAR X KISMI (Katsayıları Vinegar'a Lineer Bağlı) ---")
        for i in range(self.m):
            if self.q**i <= self.D:
                const_C = self.E.random_element()
                lin_C = [self.E.random_element() for _ in range(self.v)]
                self.Secret_Map['linear'].append((i, const_C, lin_C))

                # Print formatting
                coeff_str = f"[{const_C}"
                for k, val in enumerate(lin_C):
                    if val != 0: coeff_str += f" + ({val})*v_{k+1}"
                coeff_str += "]"
                print(f"   + {coeff_str} * X^{self.q**i}")

        # C) Gamma Kısım (Sabit Terim)
        # Yalnızca vinegar değişkenlerine bağlı sabit terim: bir sabit,
        # vinegar değişkenlerinde doğrusal bir kısım ve vinegar
        # değişkenleri arasındaki tüm ikili çarpımları (v_i * v_j, i<=j)
        # içeren kuadratik bir kısımdan oluşur. Bu terim, X'ten (Oil)
        # tamamen bağımsız olduğundan indirgeme sonrasında saf
        # Vinegar*Vinegar terimlerine karşılık gelir.
        print("\n   --- C) GAMMA KISMI (Vinegar'a Kuadratik Bağlı Sabit Terim) ---")
        gamma_const = self.E.random_element()
        gamma_lin = [self.E.random_element() for _ in range(self.v)]
        gamma_quad = []
        for i in range(self.v):
            for j in range(i, self.v):
                coeff = self.E.random_element()
                gamma_quad.append((coeff, i, j))
        self.Secret_Map['gamma'] = (gamma_const, gamma_lin, gamma_quad)

        gamma_str = f"   Gamma(V) = {gamma_const}"
        for k, val in enumerate(gamma_lin):
            if val != 0: gamma_str += f" + ({val})*v_{k+1}"
        for val, i, j in gamma_quad:
            if val != 0: gamma_str += f" + ({val})*v_{i+1}*v_{j+1}"
        print(gamma_str)

        # --- 3. AÇIK ANAHTAR İNŞASI ---
        print("\n" + ">>> BÖLÜM 2: AÇIK ANAHTAR İNŞASI (SEMBOLLERLE) <<<")

        # A) T(x) -> y
        # Girdi değişkenleri x, gizli doğrusal (afin) dönüşüm T aracılığıyla
        # y = T_mat * x + T_vec biçiminde "karıştırılır". Bu adım, saldırganın
        # açık anahtardan doğrudan orijinal x değişkenlerine erişimini
        # engelleyen ilk gizleme katmanıdır.
        x_vec = vector(self.R, self.vars)
        y_vec = self.T_mat * x_vec + self.T_vec
        print(f"[ADIM 3.1] T(x) Dönüşümü (Vektör):")
        print(f"   y = {y_vec}")

        # B) Değişken Ayrıştırma
        # y vektörünün ilk m bileşeni, {1, w, ..., w^{m-1}} bazı
        # kullanılarak SEMBOLİK biçimde tek bir genişleme cismi elemanına
        # (X_sembolik) birleştirilir; bu, F_q^m -> F_{q^m} izomorfizmasının
        # (vector_to_extension'ın sembolik analogu) polinom düzeyinde
        # uygulanmasıdır. Kalan v bileşen ise doğrudan Vinegar
        # değişkenleri (V_sembolik) olarak ayrılır.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])
        w_gen = self.E.gen()

        X_sym = sum(y_vec_E[k] * (w_gen**k) for k in range(self.m))
        V_sym = y_vec_E[self.m:]

        print(f"\n[ADIM 3.2] Değişkenlerin Ayrıştırılması (Oil ve Vinegar)")
        print(f"   Genişleme elemanı X (x değişkenlerine bağlı) şu şekildedir:")
        print(f"   X_sembolik = {X_sym}")
        print(f"   Vinegar Değişkenleri V (x değişkenlerine bağlı):")
        for k, v_poly in enumerate(V_sym):
            print(f"   v_{k+1} = {v_poly}")

        # C) F(X_sym, V_sym) -> Z
        # Gizli merkez polinom, sembolik X_sym ve V_sym ifadeleri
        # kullanılarak BÜYÜK CİSİM (F_{q^m}) üzerinde yeniden değerlendirilir.
        # Bu adım, core + linear + gamma bileşenlerinin toplanmasıyla
        # gerçekleştirilir ve merkez dönüşümün girdi değişkenleri x
        # cinsinden sembolik ifadesini (Z_sym) üretir.
        print(f"\n[ADIM 3.3] Z = F(X_sym, V_sym) Sembolik Yerine Koyma")
        print("   (Bu adımda Core, Linear ve Gamma kısımları toplanır...)")

        Z_sym = R_E_multi(0)
        # Core
        for deg, coeff in self.Secret_Map['core'].dict().items():
            Z_sym += coeff * (X_sym**deg)
        # Linear
        for i, const_C, lin_C in self.Secret_Map['linear']:
            coeff = R_E_multi(const_C)
            for k in range(self.v): coeff += lin_C[k] * V_sym[k]
            Z_sym += coeff * (X_sym**(self.q**i))
        # Gamma
        g_const, g_lin, g_quad = self.Secret_Map['gamma']
        gamma_val = R_E_multi(g_const)
        for k in range(self.v): gamma_val += g_lin[k] * V_sym[k]
        for coeff, i, j in g_quad: gamma_val += coeff * V_sym[i] * V_sym[j]
        Z_sym += gamma_val

        print(f"   -> Ham Z Polinomu Derecesi: {Z_sym.degree()} (İndirgeme öncesi)")

        # D) İndirgeme
        # Z_sym, cisim denklemleri x_i^q - x_i = 0 idealine göre indirgenir.
        # Bu adım, HFE ailesinin teorik güvencesini somutlaştırır:
        # X^{q^i+q^j} gibi görünürde yüksek dereceli terimler, bu indirgeme
        # sonrasında yalnızca KUADRATİK (en fazla ikinci dereceden) terimler
        # olarak ortaya çıkar. Elde edilen Z_reduced'ın derecesinin
        # kuadratik olması, açık anahtarın neden bir MQ (Multivariate
        # Quadratic) sistemi olduğunu doğrudan göstermektedir.
        print(f"\n[ADIM 3.4] İndirgeme (x^q - x)")
        vars_E = R_E_multi.gens()
        field_eqs = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs)
        Z_reduced = Ideal_E.reduce(Z_sym)
        # ... (Önceki kodlar: Z_reduced hesaplandıktan sonra) ...

        print(f"   -> İndirgenmiş Z Derecesi: {Z_reduced.degree()} (En fazla kuadratik olmalı)")

        # --- BU SATIRLARI EKLE ---
        print(f"   -> İndirgenmiş Z Polinomu (Açık Hali):")
        print(f"      {Z_reduced}")
        # -------------------------

        # E) Vektöre Dönüş
        # Z_reduced, E = F_{q^m} katsayılı tek bir polinom olduğundan, her
        # bir katsayı extension_to_vector() ile F_q üzerinde m boyutlu bir
        # vektöre açılır. Bu vektörün k. bileşeni, m adet F_q katsayılı
        # polinomdan (z_polys) oluşan z_vec dizisinin k. polinomuna
        # eklenir. Sonuç olarak Fbar^{-1} izomorfizmasının polinom
        # düzeyindeki karşılığı elde edilmiş olur: F_{q^m} üzerindeki tek
        # bir polinom, F_q üzerinde m adet çok değişkenli polinoma
        # dönüştürülür.
        z_polys = [self.R(0) for _ in range(self.m)]
        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff)
            monomial = self.R.monomial(*exps)
            for k in range(self.m): z_polys[k] += coeff_vec[k] * monomial
        z_vec = vector(self.R, z_polys)

        # F) S Dönüşümü (Minus)
        # Son olarak, m adet polinomdan oluşan z_vec üzerine S dönüşümü
        # uygulanarak (m-a) x m matris çarpımı + öteleme ile) boyut m'den
        # m-a'ya düşürülür. Bu, MINUS modifikasyonunun somut uygulamasıdır:
        # a adet denklem sistematik olarak "silinmiş" (saldırgana
        # gösterilmemiş) olur. Ardından cisim idealine göre son bir
        # indirgeme (reduce_F_ideal) uygulanarak açık anahtar nihai halini
        # alır.
        raw_pk = self.S_mat * z_vec + self.S_vec
        self.Public_Key = self.reduce_F_ideal(raw_pk)

        print(f"\n[ADIM 3.5] Açık Anahtar (S o Z_vec)")
        print(f"   (Toplam {len(self.Public_Key)} adet polinom)")
        for idx, p in enumerate(self.Public_Key):
            print(f"   P_{idx} = {p}")

        return self.Public_Key

    def sign(self, message_vec):
        """
        Verilen mesaj özeti (hash) vektörü için HFEv- gizli anahtarını
        kullanarak bir dijital imza üretir.

        Algoritmanın Adımları
        -----------------------
        1. Ters S İşlemi:
           S dönüşümü genel olarak örten ancak birebir OLMAYAN (boyut
           düşürücü) bir dönüşüm olduğundan, S(z) = message_vec denklemini
           sağlayan z çözümü TEK olmayabilir. Bu nedenle:
             - Doğrusal denklem sisteminin bir ÖZEL çözümü (z_particular)
               S_mat.solve_right ile bulunur.
             - S_mat'ın SAĞ ÇEKİRDEĞİ (right kernel), tüm olası çözümler
               kümesini (z_particular + kernel) parametrize eder. Bu,
               Minus modifikasyonunun getirdiği "kayıp boyut"u (a adet
               serbestlik derecesini) temsil eder.

        2. Deneme Döngüsü (en fazla 100 deneme):
           Her denemede:
             a) Minus çekirdeğinden RASTGELE bir eleman seçilerek olası
                çözümlerden biri (z_rand) belirlenir.
             b) F_q üzerinde RASTGELE vinegar değerleri (vinegar_vals)
                seçilir.
             c) z_rand, vector_to_extension() ile genişleme cismindeki
                hedef değere (W_target) dönüştürülür.
             d) Seçilen vinegar değerleri, gizli merkez polinomun linear
                ve gamma bileşenlerinde SAYISAL olarak yerine konularak,
                yalnızca X'e bağlı tek değişkenli bir polinom P_final(X)
                elde edilir (Vinegar değişkenlerinin "dondurulması").
             e) P_final(X) - W_target = 0 denkleminin genişleme cismi E
                üzerindeki KÖKLERİ hesaplanır (Sage'in .roots() metodu;
                kuramsal temelde Berlekamp gibi sonlu cisim üzerinde
                çarpanlara ayırma algoritmalarına dayanır).
             f) Bir kök bulunursa, bu kök (Oil değeri) extension_to_vector
                ile F_q vektörüne çevrilir, vinegar değerleriyle
                birleştirilerek y = (Oil || Vinegar) ara vektörü oluşturulur
                ve T dönüşümünün TERSİ uygulanarak nihai imza elde edilir:
                signature = T^{-1} * (y - T_vec).
             g) Kök bulunamazsa (P_final - W_target polinomunun E üzerinde
                kökü yoksa), farklı rastgele seçimlerle döngü tekrarlanır.

        Bu retry mekanizması, HFE ailesinin doğal bir özelliğidir: D
        derecesine sahip rastgele bir polinomun genişleme cisminde kökünün
        bulunma olasılığı kesin (1) değildir; bu nedenle vinegar/minus
        rastgeleliği ile birden fazla deneme genellikle gereklidir.

        Parametreler
        ------------
        message_vec : vector
            Uzunluğu (m - a) olan, F_q üzerinde tanımlı mesaj özeti
            (hash) vektörü.

        Dönüş Değeri
        ------------
        vector veya None
            Başarılı olursa, uzunluğu n olan imza vektörü (F_q üzerinde);
            100 deneme içinde kök bulunamazsa None döner.

        İstisnalar (Raises)
        --------------------
        ValueError
            message_vec'in uzunluğu (m - a) değilse fırlatılır.
        """
        print("\n" + ">>> BÖLÜM 3: İMZALAMA SÜRECİ DETAYI <<<")
        print(f"   Mesaj Özeti (h): {message_vec}")

        if len(message_vec) != self.m - self.a:
            raise ValueError(f"Mesaj uzunluğu {self.m - self.a} olmalıdır.")

        # 1. S^-1
        # S(z) = message_vec denklemini sağlayan z için önce özel bir
        # çözüm (z_particular), ardından tüm çözüm kümesini üreten
        # homojen sistemin çözüm uzayı (S_kernel = ker(S_mat)) hesaplanır.
        # Genel çözüm kümesi z_particular + S_kernel biçimindedir; bu,
        # Minus modifikasyonunun bıraktığı "belirsizlik" (a boyutlu
        # serbestlik) derecesini temsil eder.
        rhs = message_vec - self.S_vec
        z_particular = self.S_mat.solve_right(rhs)
        S_kernel = self.S_mat.right_kernel()

        print(f"\n[ADIM 4.1] Ters S İşlemi")
        print(f"   Özel Çözüm (z_p): {z_particular}")
        print(f"   Minus Kernel (Kayıp Boyut): {S_kernel.basis()}")

        # 2. Döngü
        # HFE tipi şemalarda merkez polinomun her deneme için genişleme
        # cisminde kökü olacağı garanti edilmediğinden, kök bulunana
        # kadar (veya deneme sınırına ulaşılana kadar) rastgele
        # vinegar/minus seçimleriyle tekrar denenir.
        attempts = 0
        while attempts < 100:
            attempts += 1
            print(f"\n   --- İMZA DENEMESİ {attempts} ---")

            # Rastgelelikler
            # Minus kernelinden rastgele bir eleman seçilerek olası z
            # çözümlerinden biri belirlenir; ayrıca F_q üzerinde rastgele
            # vinegar değerleri seçilir.
            minus_vals = S_kernel.random_element()
            z_rand = z_particular + minus_vals
            vinegar_vals = random_vector(self.F, self.v)

            print(f"   1. Rastgele Seçimler:")
            print(f"      Seçilen Vinegar Değerleri (v): {vinegar_vals}")
            print(f"      Seçilen Minus (Kernel): {minus_vals}")

            # Hedef W
            # z_rand vektörü, vector_to_extension ile genişleme cismi
            # F_{q^m} elemanına dönüştürülerek P_v(X) = W_target
            # denkleminin sağ tarafı elde edilir.
            W_target = self.vector_to_extension(z_rand)
            print(f"   2. Hedef Denklem Sağ Tarafı (W): {W_target}")

            # P_v(X) OLUŞTURMA (EN ÖNEMLİ KISIM)
            # Vinegar değerleri sayısal olarak yerine konur ve tek değişkenli pol. çıkar
            # Bu adım, gizli merkez polinomun genel (Vinegar-bağımlı)
            # formundan, seçilen vinegar_vals için TEK DEĞİŞKENLİ (yalnızca
            # X'e bağlı) somut bir polinoma indirgenmesini sağlar. Bu
            # indirgeme sayesinde, aslında çok değişkenli olan problem,
            # genişleme cismi üzerinde tek değişkenli bir polinom denklemi
            # çözme problemine dönüşür.
            P_final = self.R_E(self.Secret_Map['core'])
            X_gen = self.R_E.gen()

            # Linear Kısım Yerine Koyma
            for i, const_C, lin_C in self.Secret_Map['linear']:
                coeff = const_C
                for k in range(self.v): coeff += lin_C[k] * vinegar_vals[k]
                P_final += coeff * (X_gen**(self.q**i))

            # Gamma Kısım Yerine Koyma
            g_const, g_lin, g_quad = self.Secret_Map['gamma']
            val_gamma = g_const
            for k in range(self.v): val_gamma += g_lin[k] * vinegar_vals[k]
            for val, i, j in g_quad: val_gamma += val * vinegar_vals[i] * vinegar_vals[j]
            P_final += val_gamma

            print(f"   3. Çözülecek Polinom P_v(X) [Vinegar Yerine Kondu]:")
            print(f"      {P_final} = {W_target}")

            # Kök Bulma
            # P_final(X) - W_target = 0 denkleminin E = F_{q^m} üzerindeki
            # kökleri hesaplanır. Bu, HFEv- imzalama sürecinin hesaplama
            # açısından en kritik adımıdır; kavramsal olarak Berlekamp
            # veya Cantor-Zassenhaus gibi sonlu cisim üzerinde polinom
            # çarpanlara ayırma algoritmalarına dayanır (Sage'in .roots()
            # metodu bu algoritmaları içsel olarak kullanır).
            roots = (P_final - W_target).roots()
            print(f"   4. Kök Sayısı: {len(roots)}")

            if len(roots) > 0:
                root = roots[0][0]
                print(f"      *** KÖK BULUNDU (R): {root}")

                # İmzayı oluştur
                # Bulunan kök (Oil değeri, genişleme cismi elemanı),
                # extension_to_vector ile F_q üzerinde m boyutlu bir
                # vektöre çevrilir ve seçilen vinegar değerleriyle
                # birleştirilerek n = m + v boyutlu ara vektör y elde
                # edilir. y, aslında T(x) = y denklemindeki y'ye karşılık
                # gelir; dolayısıyla orijinal imza x, T'nin tersi
                # uygulanarak geri kazanılır.
                oil_vec = self.extension_to_vector(root)
                full_y = list(oil_vec) + list(vinegar_vals)
                y_vec = vector(self.F, full_y)
                # --- EKLENECEK SATIRLAR ---
                print(f"      -> Oil Vektörü (R'nin Vektör Hali): {oil_vec}")
                print(f"      -> Tam Ara Vektör y = (Oil || Vinegar): {y_vec}")
                # --------------------------

                # y = T(x) = T_mat * x + T_vec olduğundan, x = T_mat^{-1} *
                # (y - T_vec) ile orijinal imza (gizli anahtarla üretilmiş
                # ön-görüntü) hesaplanır.
                signature = self.T_mat.inverse() * (y_vec - self.T_vec)
                print(f"   5. İMZA (T^-1 uygulandı): {signature}")
                return signature

        return None

    def verify(self, message, signature):
        """
        Verilen (mesaj, imza) ikilisinin, HFEv- açık anahtarına göre
        geçerli olup olmadığını, ayrıntılı ara adımlar yazdırarak denetler.

        Teorik Gerekçe
        ---------------
        Doğrulama süreci, imzalamanın aksine GİZLİ anahtara ihtiyaç
        duymaz; yalnızca açık anahtar polinomlarının (self.Public_Key)
        doğrudan değerlendirilmesine dayanır. Bu asimetri (imzalamanın
        yavaş/gizli-anahtar-bağımlı, doğrulamanın hızlı/açık-anahtar-bağımlı
        olması), HFEv- gibi çok değişkenli polinom tabanlı imza
        şemalarının karakteristik özelliğidir: P(z) hesaplaması yalnızca
        polinom değerlendirmesi olduğundan, doğrulama polinom zamanda ve
        pratikte çok hızlı gerçekleştirilir.

        Algoritmanın Adımları
        -----------------------
        1. Girdi geçerliliği (mesaj ve imza uzunlukları) denetlenir.
        2. İmza vektörünün her bileşeni, ilgili değişkenlere (x_0, ..., x_{n-1})
           atanarak açık anahtarın her bir polinomu P_i için
           P_i(signature) değeri hesaplanır.
        3. Elde edilen değerler vektörü calc_vec = P(z), orijinal mesaj
           özeti (message) ile karşılaştırılır.
        4. calc_vec == message ise imza GEÇERLİ, aksi halde GEÇERSİZ
           kabul edilir.

        Parametreler
        ------------
        message : vector
            Uzunluğu (m - a) olan, doğrulanacak orijinal mesaj özeti (hash).
        signature : vector veya None
            sign() metodundan elde edilen, uzunluğu n olan imza vektörü.
            None ise (imza üretilememişse) doğrulama yapılmaz.

        Dönüş Değeri
        ------------
        bool
            İmza geçerliyse True, geçersizse veya üretilememişse False.

        İstisnalar (Raises)
        --------------------
        ValueError
            message'ın uzunluğu (m - a) değilse veya signature'ın
            uzunluğu n değilse fırlatılır.
        """
        print("\n" + ">>> BÖLÜM 4: SAĞLAMA VE DOĞRULAMA (VERIFICATION) <<<")
        print("-" * 60)
        print(f"   [Girdi] Orijinal Mesaj Özeti (h): {message}")
        print(f"   [Girdi] Hesaplanan İmza (z):      {signature}")

        if signature is None:
            print("   İmza üretilemediği için doğrulama yapılamaz.")
            return False

        if len(message) != self.m - self.a:
            raise ValueError(f"Mesaj uzunluğu {self.m - self.a} olmalıdır.")

        if len(signature) != self.n:
            raise ValueError(f"İmza uzunluğu {self.n} olmalıdır.")

        print(f"\n   [ADIM 4.1] Açık Anahtar Polinomlarında Değerlendirme P(z)")

        # İmza değerlerini değişkenlerle eşleştir (x0=..., x1=...)
        # Bu sözlük, açık anahtar polinomlarındaki her x_i değişkenini
        # imza vektörünün ilgili bileşeniyle ilişkilendirir; polinom
        # değerlendirmesi (substitution) için kullanılır.
        val_dict = {self.vars[i]: signature[i] for i in range(self.n)}

        calculated_hash = []
        print("   Polinom sonuçları hesaplanıyor:")

        # Her bir açık anahtar polinomunda yerine koy
        # Açık anahtarın her bir P_i polinomu, imza vektörü üzerinde
        # değerlendirilerek P(z) vektörünün ilgili bileşeni hesaplanır.
        for idx, poly in enumerate(self.Public_Key):
            result = poly.subs(val_dict)
            calculated_hash.append(result)
            # İstersen her satırı basabilirsin (çok yer kaplamaması için kapalı tuttum):
            # print(f"      P_{idx}(z) = {result}")

        calc_vec = vector(self.F, calculated_hash)

        print(f"   -> Hesaplanan Vektör P(z): {calc_vec}")

        print(f"\n   [ADIM 4.2] Sonuç Karşılaştırması")
        print(f"      P(z) : {calc_vec}")
        print(f"      h    : {message}")

        # Doğrulama koşulu: hesaplanan P(z) değeri, orijinal mesaj özeti
        # ile birebir eşleşiyorsa imza kriptografik olarak geçerlidir.
        if calc_vec == message:
            print("\n   [OK] SONUÇ: EŞLEŞME BAŞARILI! (İmza Matematiksel Olarak Doğru)")
            return True
        else:
            print("\n   [HATA] SONUÇ: EŞLEŞME HATASI! (İmza Geçersiz)")
            return False




In [4]:
# ============================================================================
# SENARYO OLUŞTURMA
# ============================================================================
# Parametreler: q=3, m=4, D=10, v=2, a=1
# Toplam imza değişkeni sayısı: n = m + v = 6
# Mesaj/hash boyutu: m - a = 3
#
# Bu blok, yukarıda tanımlanan HFEvminus sınıfının tam bir yaşam döngüsünü
# (anahtar üretimi -> imzalama -> doğrulama) örneklendirmek amacıyla
# çalıştırılan bir DEMO/TEST senaryosudur. 
hfev = HFEvminus(q=3, m=4, D=10, v=2, a=1)
pk = hfev.generate_keys()

# Rastgele bir mesaj/hash oluştur
msg = random_vector(hfev.F, hfev.m - hfev.a)

# İmza üret
sig = hfev.sign(msg)

# Doğrulama
if sig is not None:
    hfev.verify(msg, sig)
else:
    print("İmza oluşturulamadığı için doğrulama yapılamadı.")



HFEv- (HFE - Vinegar - Minus) TAM DETAYLI ANALİZ
[KURULUM] Parametreler: q=3, m=4, v=2, a=1, D=10
[KURULUM] İndirgenemez Polinom: X^4 + 2*X^3 + 2
--------------------------------------------------------------------------------

>>> BÖLÜM 1: GİZLİ ANAHTAR ÜRETİMİ (KEY GENERATION) <<<

[ADIM 1] Gizli Dönüşümler (S ve T)
   T Matrisi (6x6):
[1 0 2 0 2 2]
[2 1 0 2 0 1]
[1 1 2 2 2 2]
[0 1 2 1 2 1]
[1 1 0 0 2 0]
[2 2 1 2 1 2]
   T Vektörü (c_T): (2, 2, 2, 2, 1, 2)
   S Matrisi (3x4 - Minus!):
[0 2 2 2]
[0 0 2 1]
[1 1 2 1]
   S Vektörü (c_S): (1, 1, 0)

[ADIM 2] Gizli HFEv Polinomu F(X, V) Bileşenleri
   --- A) CORE KISMI (Sadece X'e bağlı terimler) ---
   F_core(X) = (w^3 + 2*w)*X^10 + (2*w^2 + 2*w + 1)*X^6 + (2*w^2 + w + 2)*X^4 + (w^3 + w^2 + 2*w + 1)*X^2

   --- B) LINEAR X KISMI (Katsayıları Vinegar'a Lineer Bağlı) ---
   + [w^2 + (2*w^3 + 2*w^2 + 2)*v_1 + (2*w^3 + 2*w^2 + 2)*v_2] * X^1
   + [w^3 + 2*w^2 + 2*w + (2*w^3 + 2*w + 1)*v_1 + (2*w^3 + 2*w^2 + w)*v_2] * X^3
   + [w^3 + 2*w^2 + 2